# Zn non-broadened vs broadened spectra

Loads matched Zn FEFF spectra from the Figshare tree, interpolates each onto the 9659–9694 eV grid (141 points, 0.25 eV), applies Gaussian broadening, and compares.

- **2×2 overlay:** four individual structures, raw vs broadened.
- **Mean of all:** average of every matched Zn spectrum, raw vs broadened.

Data root is resolved like the other committed notebooks: `$OMNIXAS_DATA_ROOT` if set, otherwise `<repo parent>/OmniXAS_data`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Resolve repo root (directory that contains the `omnixas` package).
REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir()
)
# Data root: env override, else sibling of the repo named OmniXAS_data.
DATA_ROOT = Path(
    os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")
).expanduser().resolve()

FEFF_ROOT = DATA_ROOT / "figshare" / "FEFF" / "Zn"
MANIFEST = DATA_ROOT / "figshare" / "manifest.csv"

print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("FEFF_ROOT exists:", FEFF_ROOT.is_dir())
print("manifest exists:", MANIFEST.is_file())

START_EV = 9659.0
END_EV = 9694.0
N_POINTS = 141
GRID = np.linspace(START_EV, END_EV, N_POINTS)
STEP_EV = float(GRID[1] - GRID[0])

SIGMA_EV = 0.5  # Gaussian sigma in eV; try 0.3 / 0.5 / 1.0


In [ ]:
def gaussian_kernel(sigma_ev: float, step_ev: float, radius: int = 3) -> np.ndarray:
    sigma_pt = sigma_ev / step_ev
    half = max(1, int(radius * sigma_pt))
    x = np.arange(-half, half + 1)
    k = np.exp(-0.5 * (x / sigma_pt) ** 2)
    return k / k.sum()




In [ ]:
def broaden(y: np.ndarray, sigma_ev: float = SIGMA_EV) -> np.ndarray:
    k = gaussian_kernel(sigma_ev, STEP_EV)
    half = len(k) // 2
    y_pad = np.pad(y, half, mode="edge")
    return np.convolve(y_pad, k, mode="valid")




In [ ]:
def load_spectrum(material_id: str, site: int) -> np.ndarray | None:
    path = FEFF_ROOT / material_id / "FEFF-XANES" / f"{site:03d}_Zn" / "spectrum_raw.dat"
    if not path.is_file():
        return None
    table = np.loadtxt(path, comments="#", ndmin=2)
    energy, mu = table[:, 0], table[:, 3]
    keep = np.isfinite(energy) & np.isfinite(mu) & (mu > 0)
    energy, mu = energy[keep], mu[keep]
    if energy.size < 2:
        return None
    order = np.argsort(energy, kind="stable")
    energy, mu = energy[order], mu[order]
    uniq, starts, counts = np.unique(energy, return_index=True, return_counts=True)
    if np.any(counts > 1):
        mu = np.add.reduceat(mu, starts) / counts
        energy = uniq
    inside = (energy > START_EV) & (energy < END_EV)
    if not inside.any():
        return None
    return np.interp(GRID, energy[inside], mu[inside]).astype(np.float32)




In [ ]:
df = pd.read_csv(MANIFEST)
zn = df.query("element == 'Zn' and status == 'matched'").drop_duplicates(["material_id", "site"])
print("matched Zn structures:", len(zn))

four = zn.head(4)
spectra = []
for _, row in four.iterrows():
    y = load_spectrum(row.material_id, int(row.site))
    if y is not None:
        spectra.append((f"{row.material_id} site {row.site}", y, broaden(y)))
print("loaded:", len(spectra))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True)
for ax, (label, raw, broad) in zip(axes.flat, spectra):
    ax.plot(GRID - START_EV, raw, color="black", linewidth=1, label="raw")
    ax.plot(GRID - START_EV, broad, color="tab:red", linewidth=1.4, label=f"broadened σ={SIGMA_EV} eV")
    ax.set_title(label, fontsize=9)
    ax.grid(alpha=0.2)
    ax.set_ylabel("intensity")
axes.flat[0].legend(fontsize=8, frameon=False)
fig.supxlabel("Energy above 9659 eV (eV)")
fig.suptitle(f"Individual Zn structures: raw vs broadened (σ={SIGMA_EV} eV)", fontsize=11)
fig.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
raw_list, broad_list, used = [], [], []
for _, row in zn.iterrows():
    y = load_spectrum(row.material_id, int(row.site))
    if y is None:
        continue
    raw_list.append(y)
    broad_list.append(broaden(y))
    used.append((row.material_id, int(row.site)))

raw_all = np.stack(raw_list)
broad_all = np.stack(broad_list)
mean_raw = raw_all.mean(0)
mean_broad = broad_all.mean(0)
print("spectra used:", len(used))
print("shapes:", raw_all.shape, broad_all.shape)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(GRID - START_EV, mean_raw, color="black", linewidth=1.4, label="mean raw")
axes[0].plot(GRID - START_EV, mean_broad, color="tab:red", linewidth=1.6, label=f"mean broadened σ={SIGMA_EV} eV")
axes[0].set_ylabel("intensity")
axes[0].legend(fontsize=9, frameon=False)
axes[0].grid(alpha=0.2)
axes[0].set_title(f"Mean of {len(used)} matched Zn spectra", fontsize=11)

axes[1].plot(GRID - START_EV, mean_broad - mean_raw, color="tab:green", linewidth=1, label="broadened − raw")
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_ylabel("difference")
axes[1].set_xlabel("Energy above 9659 eV (eV)")
axes[1].grid(alpha=0.2)
fig.tight_layout()
plt.show()


## How to read it

- If the red (broadened) line removes the small up-down wiggles that the black (raw) line keeps, the raw spectra carry high-frequency noise the XAS head can overfit.
- Try `SIGMA_EV = 0.3, 0.5, 1.0` to see how much smoothing removes the jaggedness without washing out the edge shape.
- If the mean broadened curve is much smoother while keeping the main peaks, training on broadened targets should give smoother predictions.

## Caveat

Broadening removes fine detail. If the wiggles are real spectral fine structure (not noise), broadening loses accuracy. The published 8-element OmniXAS data were cleaned, broadened, edge-aligned, and interpolated; your Zn targets are raw-interpolated, so the comparison below tells you whether broadening would help this dataset.